# Datasets and Dataloaders

In [4]:
from run_easg import EASGData
from dataset import myEASGDataset
from pathlib import Path
from torch_geometric.loader import DataLoader

ann_path = 'annts_in_new_format/'
with open(ann_path + 'verbs.txt') as f:
    verbs = [l.strip() for l in f.readlines()]

with open(ann_path + 'objects.txt') as f:
    objs = [l.strip() for l in f.readlines()]

with open(ann_path + 'relationships.txt') as f:
    rels = [l.strip() for l in f.readlines()]

path_annts = Path("annts_in_new_format")
path_data = Path('data')

train_original = EASGData(path_annts, path_data, 'train', verbs, objs, rels)
validation_original = EASGData(path_annts, path_data, 'val', verbs, objs, rels)

train_dataset = myEASGDataset(train_original)
validation_dataset = myEASGDataset(validation_original)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

# Training the model

In [22]:
import torch
from torch import cuda
from models import LinearProjection, myGCN, myClassifier, EdgeClassifier

obj_dim = 1024                  # original object dimension
verb_dim = 2304                 # original verb dimension
hidden_projection_dim = 1024    # hidden dimension of linear projection 
projection_dim = 1024            # final dimension after linear projection
hidden_dim = 700              # hidden dim for gnn network
output_dim = 512                # output dim for gnn network
num_rels = 13                   # number of possible relationships (possible classes)
learning_rate = 0.001
num_epochs = 100
device = 'cuda' if cuda.is_available() else print('CUDA NOT AVAILABLE')
device = 'cpu'

In [40]:
import importlib, models
importlib.reload(models)
import models

lp = LinearProjection(verb_dim, obj_dim, hidden_projection_dim, projection_dim, device)
res_lp = lp(train_dataset[0].x)
res_lp

tensor([[-0.0924, -0.0059, -0.0066,  ...,  0.0079,  0.0209, -0.0257],
        [-0.1178,  0.1860, -0.1710,  ...,  0.6331, -0.1006, -0.1987],
        [ 0.0093, -0.0087, -0.0265,  ...,  0.0466,  0.0046, -0.0684]],
       grad_fn=<CopySlices>)

In [34]:
train_dataset[0].x

tensor([[0.3380, 0.0350, 0.1268,  ..., 0.0165, 0.0483, 0.0006],
        [0.0000, 0.4803, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])